# Systems of Ordinary Differential Equations
# Systèmes d'Équations Différentielles Ordinaires

**AIMS Master's Programme — ODE Course**

We study systems of first-order ODEs $\mathbf{x}' = A\mathbf{x}$, eigenvalue analysis (analyse des valeurs propres), phase portraits (portraits de phase), and classical models from ecology and epidemiology.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from numpy.linalg import eig

plt.rcParams.update({'figure.figsize': (8, 6), 'font.size': 12})

## 1. From Higher-Order to First-Order Systems

Any $n$-th order ODE can be rewritten as a system of $n$ first-order ODEs by introducing new variables.

**Example:** The second-order equation $y'' + 3y' + 2y = 0$ becomes, with $x_1 = y$ and $x_2 = y'$:

$$\begin{pmatrix} x_1' \\ x_2' \end{pmatrix} = \begin{pmatrix} 0 & 1 \\ -2 & -3 \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix}$$

This is the standard form $\mathbf{x}' = A\mathbf{x}$. The eigenvalues of $A$ determine the qualitative behaviour.

In [ ]:
# Convert y'' + 3y' + 2y = 0 to system form and solve
A = np.array([[0, 1], [-2, -3]])
eigenvalues, eigenvectors = eig(A)
print(f"Matrix A:\n{A}")
print(f"\nEigenvalues (valeurs propres): {eigenvalues}")
print(f"Eigenvectors (vecteurs propres):\n{eigenvectors}")

# Solve the system
def linear_system(t, x):
    return A @ x

t_span = (0, 5)
t_eval = np.linspace(*t_span, 300)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Several initial conditions
ics = [[1, 0], [0, 1], [-1, 1], [1, -1], [2, 2]]
for ic in ics:
    sol = solve_ivp(linear_system, t_span, ic, t_eval=t_eval)
    ax1.plot(sol.t, sol.y[0], linewidth=1.5)
    ax2.plot(sol.y[0], sol.y[1], linewidth=1.5)

ax1.set_xlabel('$t$'); ax1.set_ylabel('$x_1(t) = y(t)$')
ax1.set_title('Time series'); ax1.grid(True, alpha=0.3)

ax2.set_xlabel('$x_1$'); ax2.set_ylabel('$x_2$')
ax2.set_title(f'Phase portrait (λ = {eigenvalues[0]:.1f}, {eigenvalues[1]:.1f}): stable node')
ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# Figure: Stable node — both eigenvalues are real and negative.

## 2. Phase Portrait Gallery (Galerie de portraits de phase)

The eigenvalues of $A$ classify the equilibrium at the origin:

| Eigenvalues | Type | Stability |
|:---|:---|:---|
| Real, both negative | Stable node (noeud stable) | Asymptotically stable |
| Real, both positive | Unstable node (noeud instable) | Unstable |
| Real, opposite signs | Saddle point (col) | Unstable |
| Complex, Re < 0 | Stable spiral (foyer stable) | Asymptotically stable |
| Complex, Re > 0 | Unstable spiral (foyer instable) | Unstable |
| Purely imaginary | Center (centre) | Stable (not asymptotically) |

In [ ]:
def plot_phase_portrait(A, title, ax, xlim=(-3, 3), ylim=(-3, 3)):
    """Draw phase portrait for x' = Ax with streamlines and eigenvalue info."""
    vals, vecs = eig(A)
    
    # Vector field
    x = np.linspace(*xlim, 20)
    y = np.linspace(*ylim, 20)
    X, Y = np.meshgrid(x, y)
    U = A[0, 0]*X + A[0, 1]*Y
    V = A[1, 0]*X + A[1, 1]*Y
    
    ax.streamplot(X, Y, U, V, density=1.5, color='steelblue', linewidth=0.8)
    
    # Plot trajectories from several ICs
    for ic in [[2, 0], [-2, 0], [0, 2], [0, -2], [2, 2], [-2, -2]]:
        sol = solve_ivp(lambda t, x: A @ x, (0, 10), ic,
                        t_eval=np.linspace(0, 10, 500))
        ax.plot(sol.y[0], sol.y[1], 'r-', linewidth=1, alpha=0.7)
    
    ax.plot(0, 0, 'ko', markersize=6)
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.set_title(f'{title}\nλ = {vals[0]:.2f}, {vals[1]:.2f}')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

# Gallery of phase portraits
matrices = {
    'Stable node\n(noeud stable)': np.array([[-2, 0], [0, -1]]),
    'Saddle point\n(col)': np.array([[1, 0], [0, -2]]),
    'Stable spiral\n(foyer stable)': np.array([[-0.5, 2], [-2, -0.5]]),
    'Center\n(centre)': np.array([[0, 1], [-1, 0]]),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, (name, mat) in zip(axes.flat, matrices.items()):
    plot_phase_portrait(mat, name, ax)

plt.suptitle('Phase Portrait Gallery', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
# Figure: Four canonical phase portraits for linear 2D systems.

## 3. SIR Epidemiological Model (Modèle épidémiologique SIR)

The SIR model partitions a population of size $N$ into:
- $S$: Susceptible, $I$: Infected, $R$: Recovered

$$\frac{dS}{dt} = -\beta S I, \quad \frac{dI}{dt} = \beta S I - \gamma I, \quad \frac{dR}{dt} = \gamma I$$

This is a **nonlinear** system. The basic reproduction number $R_0 = \beta S_0 / \gamma$ determines epidemic threshold: an outbreak occurs when $R_0 > 1$.

In [ ]:
def sir(t, y, beta, gamma):
    S, I, R = y
    return [-beta*S*I, beta*S*I - gamma*I, gamma*I]

N = 1000
I0 = 5
S0 = N - I0
R0_init = 0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Different transmission rates
gamma = 0.1
for beta_val, style in [(0.0003, '--'), (0.0005, '-'), (0.001, '-')]:
    R0_val = beta_val * S0 / gamma
    sol = solve_ivp(sir, (0, 200), [S0, I0, R0_init],
                    args=(beta_val, gamma), t_eval=np.linspace(0, 200, 500))
    axes[0].plot(sol.t, sol.y[1], style, linewidth=2,
                label=f'β={beta_val}, $R_0$={R0_val:.1f}')

axes[0].set_xlabel('Time (days)'); axes[0].set_ylabel('Infected $I(t)$')
axes[0].set_title('SIR: Effect of transmission rate on epidemic curve')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Phase portrait: S-I plane
beta_pp = 0.0005
for I0_val in [1, 5, 20, 50]:
    sol = solve_ivp(sir, (0, 300), [N - I0_val, I0_val, 0],
                    args=(beta_pp, gamma), t_eval=np.linspace(0, 300, 1000))
    axes[1].plot(sol.y[0], sol.y[1], linewidth=1.5, label=f'$I_0={I0_val}$')

axes[1].axvline(x=gamma/beta_pp, color='gray', linestyle=':', label=f'$S^* = \\gamma/\\beta = {gamma/beta_pp:.0f}$')
axes[1].set_xlabel('Susceptible $S$'); axes[1].set_ylabel('Infected $I$')
axes[1].set_title('SIR phase portrait ($S$-$I$ plane)')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# Figure: Left — epidemic curves for different R_0. Right — S-I phase plane
# showing that I peaks when S = gamma/beta.

## 4. Lotka-Volterra Predator-Prey Model (Modèle proie-prédateur)

The classic Lotka-Volterra equations model the interaction between prey ($x$) and predator ($y$) populations:

$$\frac{dx}{dt} = \alpha x - \beta x y \quad \text{(prey growth minus predation)}$$
$$\frac{dy}{dt} = \delta x y - \gamma y \quad \text{(predator growth from feeding minus death)}$$

This system has a non-trivial equilibrium (point d'équilibre) at $(x^*, y^*) = (\gamma/\delta, \alpha/\beta)$ which is a **center** — solutions are periodic orbits.

In [ ]:
def lotka_volterra(t, z, alpha, beta, delta, gamma):
    x, y = z
    return [alpha*x - beta*x*y, delta*x*y - gamma*y]

alpha, beta, delta, gamma_lv = 1.0, 0.1, 0.075, 1.5
x_eq = gamma_lv / delta
y_eq = alpha / beta
print(f"Equilibrium point: ({x_eq:.1f}, {y_eq:.1f})")

t_span_lv = (0, 40)
t_eval_lv = np.linspace(*t_span_lv, 2000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time series
sol_lv = solve_ivp(lotka_volterra, t_span_lv, [10, 5],
                   args=(alpha, beta, delta, gamma_lv),
                   t_eval=t_eval_lv, max_step=0.05)
axes[0].plot(sol_lv.t, sol_lv.y[0], 'b-', label='Prey $x(t)$', linewidth=2)
axes[0].plot(sol_lv.t, sol_lv.y[1], 'r-', label='Predator $y(t)$', linewidth=2)
axes[0].set_xlabel('Time'); axes[0].set_ylabel('Population')
axes[0].set_title('Lotka-Volterra: Time Series')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Phase portrait with several ICs
for ic in [[10, 5], [15, 3], [30, 5], [5, 15]]:
    sol = solve_ivp(lotka_volterra, t_span_lv, ic,
                    args=(alpha, beta, delta, gamma_lv),
                    t_eval=t_eval_lv, max_step=0.05)
    axes[1].plot(sol.y[0], sol.y[1], linewidth=1.5)

axes[1].plot(x_eq, y_eq, 'k*', markersize=12, label=f'Equilibrium ({x_eq:.0f}, {y_eq:.0f})')
axes[1].set_xlabel('Prey $x$'); axes[1].set_ylabel('Predator $y$')
axes[1].set_title('Lotka-Volterra: Phase Portrait (closed orbits)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# Figure: Left — oscillatory predator-prey dynamics. Right — closed orbits
# around the equilibrium confirm it is a center.

## 5. Exercise: Competing Species Model (Modèle de compétition)

Two species compete for the same resources:

$$\frac{dx}{dt} = r_1 x\left(1 - \frac{x + \alpha_{12} y}{K_1}\right)$$
$$\frac{dy}{dt} = r_2 y\left(1 - \frac{y + \alpha_{21} x}{K_2}\right)$$

where $K_i$ are carrying capacities (capacités de charge) and $\alpha_{ij}$ are competition coefficients.

**Tasks:**
1. Find all equilibrium points (there can be up to 4).
2. Implement the system with parameters $r_1 = r_2 = 1$, $K_1 = K_2 = 100$.
3. For **coexistence** ($\alpha_{12} = \alpha_{21} = 0.5$), plot the phase portrait and verify that both species persist.
4. For **competitive exclusion** ($\alpha_{12} = 1.5, \alpha_{21} = 0.5$), show that species 2 drives species 1 to extinction.
5. Draw the **nullclines** (isoclines nulles): curves where $dx/dt = 0$ and $dy/dt = 0$. Their intersections are the equilibria.

In [ ]:
# Competing species model — starter code
def competing_species(t, z, r1, r2, K1, K2, a12, a21):
    x, y = z
    dxdt = r1 * x * (1 - (x + a12 * y) / K1)
    dydt = r2 * y * (1 - (y + a21 * x) / K2)
    return [dxdt, dydt]

r1, r2 = 1.0, 1.0
K1, K2 = 100.0, 100.0

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Case 1: Coexistence
a12, a21 = 0.5, 0.5
for ic in [[10, 90], [90, 10], [10, 10], [80, 80], [50, 5]]:
    sol = solve_ivp(competing_species, (0, 30), ic,
                    args=(r1, r2, K1, K2, a12, a21),
                    t_eval=np.linspace(0, 30, 500))
    axes[0].plot(sol.y[0], sol.y[1], linewidth=1.5)
    axes[0].plot(sol.y[0][0], sol.y[1][0], 'o', markersize=5)

# Nullclines for coexistence case
x_nc = np.linspace(0, 120, 200)
axes[0].plot(x_nc, (K1 - x_nc) / a12, 'b--', alpha=0.5, label='$dx/dt=0$')
axes[0].plot(x_nc, K2 - a21 * x_nc, 'r--', alpha=0.5, label='$dy/dt=0$')
axes[0].set_xlim(0, 120); axes[0].set_ylim(0, 120)
axes[0].set_xlabel('Species 1 ($x$)'); axes[0].set_ylabel('Species 2 ($y$)')
axes[0].set_title(f'Coexistence ($\\alpha_{{12}}={a12}, \\alpha_{{21}}={a21}$)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Case 2: Competitive exclusion
a12_ex, a21_ex = 1.5, 0.5
for ic in [[10, 90], [90, 10], [10, 10], [80, 80], [50, 5]]:
    sol = solve_ivp(competing_species, (0, 30), ic,
                    args=(r1, r2, K1, K2, a12_ex, a21_ex),
                    t_eval=np.linspace(0, 30, 500))
    axes[1].plot(sol.y[0], sol.y[1], linewidth=1.5)
    axes[1].plot(sol.y[0][0], sol.y[1][0], 'o', markersize=5)

axes[1].plot(x_nc, (K1 - x_nc) / a12_ex, 'b--', alpha=0.5, label='$dx/dt=0$')
axes[1].plot(x_nc, K2 - a21_ex * x_nc, 'r--', alpha=0.5, label='$dy/dt=0$')
axes[1].set_xlim(0, 120); axes[1].set_ylim(0, 120)
axes[1].set_xlabel('Species 1 ($x$)'); axes[1].set_ylabel('Species 2 ($y$)')
axes[1].set_title(f'Competitive exclusion ($\\alpha_{{12}}={a12_ex}, \\alpha_{{21}}={a21_ex}$)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# Figure: Left — coexistence: trajectories converge to interior equilibrium.
# Right — competitive exclusion: species 2 wins.